# Zimbabwe 2022–2023: data overview

One observation is one field in one harvest year. This notebook reads the curated DuckDB file **without changing it**, inventories the available inputs, and constructs aligned features (`X`) and maize yields (`y`). All cohort records remain available; the project's modelling subset is identified separately.

The file in this checkout is `data/cropyield.duckdb` (rather than `dataset/cropyield.duckdb`). This notebook lives in the requested `notebook/` directory; the existing guide is in `notebooks/`.


In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sql/schema.sql").exists())
DB = ROOT / "data/cropyield.duckdb"
assert DB.is_file(), DB
con = duckdb.connect(str(DB), read_only=True)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 100)
print(f"Database: {DB}")
display(con.execute("SELECT built_at, git_commit, n_labels, n_embeddings, mosaiks_features FROM build_info").df())


Database: /Users/riessarius/Documents/2026/UCSD Course/FA26/DSC 291 DSSD/Project/data/cropyield.duckdb


,built_at,git_commit,n_labels,n_embeddings,mosaiks_features
0,2026-10-30 21:53:55.201397,96e2e81af739fab7a6d3ce054177941778868260,11364,11364,1024


## 1. Locate the Zimbabwe cohort

`year` is the **harvest year**. The combined cohort name contains both years; it does not mean that every observation spans both harvests. The 2023 harvest window is October 2022–June 2023; the 2022 harvest window is October 2021–June 2022.


In [2]:
display(con.execute("""
    SELECT cohort, country, year, count(*) AS observations,
           count(*) FILTER (WHERE include_primary) AS primary_observations
    FROM labels GROUP BY ALL ORDER BY country, year
""").df())

COHORT = "zimbabwe_lacuna_2022_2023"
zimbabwe = con.execute("""
    SELECT * FROM master
    WHERE cohort = ? AND country = 'zimbabwe' AND year IN (2022, 2023)
    ORDER BY year, obs_id
""", [COHORT]).df().set_index("obs_id")
assert zimbabwe.index.is_unique

display(zimbabwe.groupby("year").agg(
    observations=("year", "size"), primary_observations=("include_primary", "sum"),
    season_start=("season_start", "min"), season_end=("season_end", "max"),
    min_clear_months=("months_with_data", "min"), max_clear_months=("months_with_data", "max")))
display(zimbabwe.groupby(["year", "include_primary", "exclusion_reason"], dropna=False)
        .size().rename("observations").reset_index())


,cohort,country,year,observations,primary_observations
0,kenya_oaf_2019_2020,kenya,2019,1768,1768
1,kenya_oaf_2019_2020,kenya,2020,2209,2209
2,malawi_2022,malawi,2022,529,0
3,malawi_2023,malawi,2023,260,260
4,zimbabwe_lacuna_2022_2023,zimbabwe,2022,169,0
5,zimbabwe_lacuna_2022_2023,zimbabwe,2023,2097,1880
6,zimbabwe_2024,zimbabwe,2024,4332,2458


,observations,primary_observations,season_start,season_end,min_clear_months,max_clear_months
year,,,,,,
2022,169,0,2021-10-01,2022-06-30,3,4
2023,2097,1880,2022-10-01,2023-06-30,4,7


,year,include_primary,exclusion_reason,observations
0,2022,False,dropped_year_2022_imagery_gap,169
1,2023,False,lacuna_intercropped,212
2,2023,False,lacuna_irrigated,5
3,2023,True,,1880


## 2. Yield target (`y`)

`yield_kg_ha` is maize yield in **kg/ha**. `yield_t_ha = yield_kg_ha / 1000` is the same target in **t/ha**. `source_yield` is another representation of the target, so it must never enter `X`. The provider field is recorded as `Yield(Mt/H`, with source units `t/ha`.

The project documents the Lacuna measurement as one 8 × 5 m crop cut, with dry weight converted to t/ha; its unit interpretation is provisional and the moisture basis is unspecified. See `docs/DATA.md`, “How each cohort measures yield”.


In [3]:
y_all = zimbabwe["yield_kg_ha"].rename("yield_kg_ha")
y_all_t_ha = (y_all / 1000).rename("yield_t_ha")
yields = zimbabwe[["year", "source_id", "yield_kg_ha", "source_yield", "source_yield_units",
                     "yield_field", "include_primary"]].assign(yield_t_ha=y_all_t_ha)
assert np.isfinite(y_all).all()
assert np.allclose(y_all, zimbabwe["source_yield"] * 1000)
print(f"All yield values are in `yields`: {len(yields):,} rows. First 10:")
display(yields.head(10))
display(yields.groupby("year")["yield_kg_ha"].describe())
display(yields.groupby("year").agg(
    missing_y=("yield_kg_ha", lambda s: s.isna().sum()),
    zero_y=("yield_kg_ha", lambda s: s.eq(0).sum()),
    mean_t_ha=("yield_t_ha", "mean")))


All yield values are in `yields`: 2,266 rows. First 10:


,year,source_id,yield_kg_ha,source_yield,source_yield_units,yield_field,include_primary,yield_t_ha
obs_id,,,,,,,,
zimbabwe:01b57ca9-cf36-4e3a-b2c0-31d76823655c,2022,01b57ca9-cf36-4e3a-b2c0-31d76823655c,1319.375,1.319375,t/ha,Yield(Mt/H,False,1.319375
zimbabwe:01b8d978-8bfe-4cbe-aac7-c54b8f4bd6cd,2022,01b8d978-8bfe-4cbe-aac7-c54b8f4bd6cd,5973.750,5.973750,t/ha,Yield(Mt/H,False,5.973750
zimbabwe:025e133b-d8b0-4cf0-add9-f36bbb4004c8,2022,025e133b-d8b0-4cf0-add9-f36bbb4004c8,112.500,0.112500,t/ha,Yield(Mt/H,False,0.112500
zimbabwe:03a84963-027a-4b4d-b817-948b0cbc4de8,2022,03a84963-027a-4b4d-b817-948b0cbc4de8,855.000,0.855000,t/ha,Yield(Mt/H,False,0.855000
zimbabwe:0a1ae3d7-100b-4005-a8f4-6e754a24451e,2022,0a1ae3d7-100b-4005-a8f4-6e754a24451e,1196.875,1.196875,t/ha,Yield(Mt/H,False,1.196875
zimbabwe:0ab223b6-2376-4c28-b8f6-843c3595493b,2022,0ab223b6-2376-4c28-b8f6-843c3595493b,4651.250,4.651250,t/ha,Yield(Mt/H,False,4.651250
zimbabwe:0cb872bd-4d7a-4bff-99c2-d1dfca4d30b4,2022,0cb872bd-4d7a-4bff-99c2-d1dfca4d30b4,4493.750,4.493750,t/ha,Yield(Mt/H,False,4.493750
zimbabwe:0f30791b-af0c-4a68-a645-7ea8285191a7,2022,0f30791b-af0c-4a68-a645-7ea8285191a7,1246.875,1.246875,t/ha,Yield(Mt/H,False,1.246875
zimbabwe:1032aa78-ba87-442c-bb6a-47cb3e907303,2022,1032aa78-ba87-442c-bb6a-47cb3e907303,81.250,0.081250,t/ha,Yield(Mt/H,False,0.081250


,count,mean,std,min,25%,50%,75%,max
year,,,,,,,,
2022,169.0,2457.224852,2259.399621,53.125,600.000,1460.000,4137.50,8833.75
2023,2097.0,3330.897830,2016.441597,0.000,1785.625,3023.125,4691.25,12430.00


,missing_y,zero_y,mean_t_ha
year,,,
2022,0,0,2.457225
2023,0,28,3.330898


## 3. All available features and supporting fields

`master` provides one row per field-season; `monthly` provides one row per field-month. Satellite inputs are alternative representations of the same imagery. The dictionary below lists **every column** in these views and separates predictors, labels, identifiers, and quality information. Coordinates and intercropping are optional non-image inputs. Other metadata need a prediction-time availability decision before use; identifiers, source yields and inclusion decisions are not predictors.


In [4]:
INDEX_NAMES = ["ndvi", "gcvi", "mtci", "ndvi705", "ndvi740"]
SEASON_FEATURES = [f"{name}_season_{stat}" for name in INDEX_NAMES for stat in ["max", "mean"]]
BANDS = {
    "b02": "Blue", "b03": "Green", "b04": "Red", "b05": "Red edge 705 nm",
    "b06": "Red edge 740 nm", "b07": "Red edge 783 nm", "b08": "Near infrared",
    "b8a": "Narrow near infrared 865 nm", "b11": "Shortwave infrared 1610 nm",
    "b12": "Shortwave infrared 2190 nm"}
MONTHLY_FEATURES = [*BANDS, *INDEX_NAMES]
VECTORS = {"presto_embedding": "presto", "mosaiks_chip": "mosaiks_chip", "mosaiks_field": "mosaiks_field"}

roles = {}
for name in SEASON_FEATURES:
    roles[name] = ("Satellite X: seasonal index", "Season maximum or mean; missing months skipped")
for name, meaning in BANDS.items():
    roles[name] = ("Satellite X: monthly band", f"{meaning}; field-median surface reflectance")
for name in INDEX_NAMES:
    roles[name] = ("Satellite X: monthly index", "Vegetation index from monthly bands")
for name in VECTORS:
    roles[name] = ("Satellite X: vector", "Frozen Presto embedding" if name == "presto_embedding" else
                   "MOSAIKS vector of full chip" if name == "mosaiks_chip" else "MOSAIKS vector of field pixels")
for name in ["yield_kg_ha", "source_yield", "source_yield_units", "yield_field"]:
    roles[name] = ("Target / target provenance; exclude from X", "Yield or its source representation")
for name in ["obs_id", "cohort", "dataset", "source_id", "source_row", "country", "year", "month"]:
    roles[name] = ("Identifier / grouping / time", "Join, cohort filter, harvest year or sequence position")
for name in ["include_primary", "exclusion_reason"]:
    roles[name] = ("Sample selection; exclude from X", "Project inclusion decision")
for name in ["latitude", "longitude", "intercropping", "field_area_ha"]:
    roles[name] = ("Optional non-image X", "WGS84 coordinate" if name in ["latitude", "longitude"] else
                   "Provider management flag" if name == "intercropping" else "Field area, hectares")
for name in ["has_imagery", "download_status", "months_with_data", "mosaiks_months_used", "coverage_state"]:
    roles[name] = ("Coverage / quality", "Availability diagnostic; keep separate from satellite signal")
for name in ["geometry_type", "field_polygon", "label_date", "season_start", "season_end"]:
    roles[name] = ("Spatial / calendar metadata", "Check prediction-time availability before treating as X")

schema = con.execute("""
    SELECT table_name, column_name, data_type FROM information_schema.columns
    WHERE table_schema = 'main' AND table_name IN ('master', 'monthly')
    ORDER BY table_name, ordinal_position
""").df()
assert set(schema.column_name) <= roles.keys()
schema["role"] = schema.column_name.map(lambda c: roles[c][0])
schema["meaning"] = schema.column_name.map(lambda c: roles[c][1])
for view, columns in schema.groupby("table_name", sort=False):
    print(f"{view}: every column ({len(columns)})")
    display(columns.drop(columns="table_name").reset_index(drop=True))


master: every column (39)


,column_name,data_type,role,meaning
0,obs_id,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
1,cohort,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
2,dataset,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
3,source_id,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
4,source_row,INTEGER,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
5,country,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
6,year,INTEGER,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
7,yield_kg_ha,DOUBLE,Target / target provenance; exclude from X,Yield or its source representation
8,source_yield,DOUBLE,Target / target provenance; exclude from X,Yield or its source representation
9,source_yield_units,VARCHAR,Target / target provenance; exclude from X,Yield or its source representation


monthly: every column (22)


,column_name,data_type,role,meaning
0,obs_id,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
1,cohort,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
2,month,VARCHAR,Identifier / grouping / time,"Join, cohort filter, harvest year or sequence position"
3,coverage_state,VARCHAR,Coverage / quality,Availability diagnostic; keep separate from satellite signal
4,b02,DOUBLE,Satellite X: monthly band,Blue; field-median surface reflectance
5,b03,DOUBLE,Satellite X: monthly band,Green; field-median surface reflectance
6,b04,DOUBLE,Satellite X: monthly band,Red; field-median surface reflectance
7,b05,DOUBLE,Satellite X: monthly band,Red edge 705 nm; field-median surface reflectance
8,b06,DOUBLE,Satellite X: monthly band,Red edge 740 nm; field-median surface reflectance
9,b07,DOUBLE,Satellite X: monthly band,Red edge 783 nm; field-median surface reflectance


The five index definitions are NDVI = (B08 − B04)/(B08 + B04), GCVI = B08/B03 − 1, MTCI = (B08 − B05)/(B05 − B04), NDVI705 = (B08 − B05)/(B08 + B05), and NDVI740 = (B08 − B06)/(B08 + B06). The seasonal summaries take each index's maximum and mean over available months.

The remaining data tables retain imagery provenance and quality fields. Their full column inventory and cohort row counts follow. Raw image pixels are stored outside DuckDB; paths in `downloads.record_dir` identify their expected location and do not establish that the image files are on this machine. No weather or soil table is present in this database.


In [5]:
tables = con.execute("""
    SELECT table_name, table_type FROM information_schema.tables
    WHERE table_schema = 'main' ORDER BY table_name
""").df()
counts = []
for table in tables.table_name:
    if table == "build_info":
        n = con.execute("SELECT count(*) FROM build_info").fetchone()[0]
    else:
        n = con.execute(f"""SELECT count(*) FROM "{table}" t
            JOIN labels l USING (obs_id)
            WHERE l.cohort = ? AND l.year IN (2022, 2023)""", [COHORT]).fetchone()[0]
    counts.append(n)
tables["cohort_rows_or_build_rows"] = counts
display(tables)

auxiliary_schema = con.execute("""
    SELECT table_name, column_name, data_type FROM information_schema.columns
    WHERE table_schema = 'main' AND table_name IN ('downloads', 'scenes', 'coverage')
    ORDER BY table_name, ordinal_position
""").df()
display(auxiliary_schema)
nonimage_columns = ["year", "latitude", "longitude", "intercropping", "field_area_ha", "geometry_type",
                    "label_date", "season_start", "season_end", "field_polygon"]
nonimage = zimbabwe[nonimage_columns].copy()
display(pd.DataFrame({"dtype": nonimage.dtypes.astype(str), "non_null": nonimage.notna().sum(),
                      "missing": nonimage.isna().sum(), "distinct_non_null": nonimage.nunique()}))


,table_name,table_type,cohort_rows_or_build_rows
0,build_info,BASE TABLE,1
1,coverage,BASE TABLE,20394
2,downloads,BASE TABLE,2266
3,labels,BASE TABLE,2266
4,master,VIEW,2266
5,monthly,VIEW,20394
6,monthly_bands,BASE TABLE,20394
7,mosaiks_monthly,BASE TABLE,13989
8,mosaiks_season,BASE TABLE,2266
9,presto_embeddings,BASE TABLE,2266


,table_name,column_name,data_type
0,coverage,obs_id,VARCHAR
1,coverage,month,VARCHAR
2,coverage,state,VARCHAR
3,coverage,item_id,VARCHAR
4,downloads,obs_id,VARCHAR
5,downloads,record_dir,VARCHAR
6,downloads,status,VARCHAR
7,downloads,aoi_type,VARCHAR
8,downloads,aoi_pixels,INTEGER
9,downloads,field_pixels,INTEGER


,dtype,non_null,missing,distinct_non_null
year,int32,2266,0,2
latitude,float64,2266,0,2258
longitude,float64,2266,0,2258
intercropping,str,2266,0,2
field_area_ha,float64,0,2266,0
geometry_type,str,2266,0,1
label_date,datetime64[us],2266,0,164
season_start,datetime64[us],2266,0,2
season_end,datetime64[us],2266,0,2
field_polygon,object,0,2266,0


## 4. Seasonal `X` and aligned `y` for every record

`X_all` combines all 10 seasonal indices, all 128 Presto dimensions and both 1,024-dimensional MOSAIKS vectors: **2,186 numeric features** per observation in this copy. These feature families can also be used separately via `X_families`. Coordinates and management fields remain in `nonimage`. All arrays are expanded explicitly; no identifier, yield, or selection flag enters `X_all`.


In [6]:
X_families = {"season_indices": zimbabwe[SEASON_FEATURES].astype(float)}
for column, prefix in VECTORS.items():
    values = zimbabwe[column]
    widths = {len(v) for v in values if isinstance(v, (list, tuple, np.ndarray))}
    assert len(widths) == 1, (column, widths)
    width = widths.pop()
    matrix = np.full((len(values), width), np.nan)
    for i, value in enumerate(values):
        if isinstance(value, (list, tuple, np.ndarray)):
            matrix[i] = value
    X_families[column] = pd.DataFrame(matrix, index=zimbabwe.index,
                                    columns=[f"{prefix}_{j:04d}" for j in range(width)])

X_all = pd.concat(X_families.values(), axis=1)
assert X_all.index.equals(y_all.index) and X_all.columns.is_unique
assert set(X_all.columns).isdisjoint(["yield_kg_ha", "source_yield", "include_primary"])
feature_summary = []
for year in sorted(zimbabwe.year.unique()):
    ids = zimbabwe.index[zimbabwe.year.eq(year)]
    for family, frame in X_families.items():
        values = frame.loc[ids].to_numpy()
        feature_summary.append({"year": year, "family": family, "rows": len(ids), "dimensions": frame.shape[1],
                                "rows_fully_finite": int(np.isfinite(values).all(axis=1).sum()),
                                "nonfinite_values": int((~np.isfinite(values)).sum())})
display(pd.DataFrame(feature_summary))
print(f"X_all: {X_all.shape}; y_all: {y_all.shape} (kg/ha); aligned by obs_id.")
print("Every scalar feature name is available in X_all.columns; previews below show the index features.")
display(X_all[SEASON_FEATURES].head())
display(X_all.iloc[:3, [10, 11, 137, 138, 1161, 1162, 2185]])


,year,family,rows,dimensions,rows_fully_finite,nonfinite_values
0,2022,season_indices,169,10,169,0
1,2022,presto_embedding,169,128,169,0
2,2022,mosaiks_chip,169,1024,169,0
3,2022,mosaiks_field,169,1024,169,0
4,2023,season_indices,2097,10,2097,0
5,2023,presto_embedding,2097,128,2097,0
6,2023,mosaiks_chip,2097,1024,2097,0
7,2023,mosaiks_field,2097,1024,2097,0


X_all: (2266, 2186); y_all: (2266,) (kg/ha); aligned by obs_id.
Every scalar feature name is available in X_all.columns; previews below show the index features.


,ndvi_season_max,ndvi_season_mean,gcvi_season_max,gcvi_season_mean,mtci_season_max,mtci_season_mean,ndvi705_season_max,ndvi705_season_mean,ndvi740_season_max,ndvi740_season_mean
obs_id,,,,,,,,,,
zimbabwe:01b57ca9-cf36-4e3a-b2c0-31d76823655c,0.739884,0.315756,3.841823,1.723864,3.271116,2.115757,0.482987,0.197339,0.109837,0.075445
zimbabwe:01b8d978-8bfe-4cbe-aac7-c54b8f4bd6cd,0.463773,0.262092,1.465692,1.292046,3.400357,1.997802,0.324209,0.161408,0.068831,0.052081
zimbabwe:025e133b-d8b0-4cf0-add9-f36bbb4004c8,0.205957,0.178818,1.315359,1.261466,1.902548,1.757909,0.126055,0.107056,0.090629,0.065460
zimbabwe:03a84963-027a-4b4d-b817-948b0cbc4de8,0.227891,0.181683,1.445084,1.340420,1.276813,0.982917,0.116171,0.082979,0.046025,0.039731
zimbabwe:0a1ae3d7-100b-4005-a8f4-6e754a24451e,0.259770,0.194905,1.027951,0.896917,2.500377,1.923703,0.172739,0.119844,0.071998,0.059742


,presto_0000,presto_0001,presto_0127,mosaiks_chip_0000,mosaiks_chip_1023,mosaiks_field_0000,mosaiks_field_1023
obs_id,,,,,,,
zimbabwe:01b57ca9-cf36-4e3a-b2c0-31d76823655c,-1.664267,-5.355895,1.119914,0.65277,0.082924,1.177724,0.000000
zimbabwe:01b8d978-8bfe-4cbe-aac7-c54b8f4bd6cd,-1.321465,-5.013140,1.157287,3.66348,5.090999,3.540726,4.643887
zimbabwe:025e133b-d8b0-4cf0-add9-f36bbb4004c8,-1.254836,-4.862906,1.475579,0.00000,0.007500,0.000000,0.000000


## 5. Monthly inputs and missing months

`monthly` contains 10 bands, 5 indices and two MOSAIKS vectors per field-month; `yield_kg_ha` is repeated there and must be removed from monthly predictors. `X_monthly` retains the 15 numeric band/index columns on an `(obs_id, month)` index. `X_monthly_vectors` retains the two vector columns separately to avoid unnecessarily expanding the whole time series. Coverage and pixel diagnostics are in `monthly_quality`.

A missing month remains a missing month; this overview performs no imputation. Monthly rows from the same field-season share one target and must stay together when splitting data.


In [7]:
monthly = con.execute("""
    SELECT m.*, l.year, b.field_pixels, b.missing
    FROM monthly m JOIN labels l USING (obs_id)
    LEFT JOIN monthly_bands b USING (obs_id, month)
    WHERE l.cohort = ? AND l.year IN (2022, 2023)
    ORDER BY m.obs_id, m.month
""", [COHORT]).df().set_index(["obs_id", "month"])
assert monthly.index.is_unique
X_monthly = monthly[MONTHLY_FEATURES].copy()
X_monthly_vectors = monthly[["mosaiks_chip", "mosaiks_field"]].copy()
monthly_quality = monthly[["year", "coverage_state", "field_pixels", "missing"]].copy()
y_monthly = monthly["yield_kg_ha"].copy()
assert np.allclose(y_monthly, y_all.reindex(monthly.index.get_level_values("obs_id")))
print(f"X_monthly: {X_monthly.shape}; monthly MOSAIKS arrays: {X_monthly_vectors.shape}")
display(X_monthly.head(9))
monthly_missingness = X_monthly.isna().groupby(monthly.year.to_numpy()).sum().T
monthly_missingness.columns.name = "harvest_year"
display(monthly_missingness.rename_axis("feature").reset_index())

monthly_vectors_summary = []
for year, frame in monthly.groupby("year"):
    for column in ["mosaiks_chip", "mosaiks_field"]:
        vectors = frame[column].map(lambda v: isinstance(v, (list, tuple, np.ndarray)))
        widths = sorted({len(v) for v in frame.loc[vectors, column]})
        nonfinite = sum(int((~np.isfinite(v)).sum()) for v in frame.loc[vectors, column])
        monthly_vectors_summary.append({"year": year, "family": column, "field_months": len(frame),
                                        "present_vectors": int(vectors.sum()), "dimensions": str(widths),
                                        "nonfinite_inside_vectors": nonfinite})
display(pd.DataFrame(monthly_vectors_summary))
coverage = monthly.reset_index().groupby(["year", "month", "coverage_state"]).size().unstack(fill_value=0)
display(coverage)


X_monthly: (20394, 15); monthly MOSAIKS arrays: (20394, 2)


b02     b03     b04  \
obs_id                                        month                             
zimbabwe:0002e51c-5124-4614-bb1b-1c1ae307de80 2022-10     NaN     NaN     NaN   
                                              2022-11     NaN     NaN     NaN   
                                              2022-12  0.0522  0.0736  0.0682   
                                              2023-01  0.0499  0.0774  0.0949   
                                              2023-02  0.0451  0.0620  0.0850   
                                              2023-03  0.0319  0.0522  0.0602   
                                              2023-04  0.0396  0.0552  0.0586   
                                              2023-05  0.0493  0.0700  0.0974   
                                              2023-06  0.0603  0.0746  0.1036   

                                                            b05       b06  \
obs_id                                        month                         
zimbabwe:0002e51c-5124-4614-bb1b-1c1ae307de80 2022-10       NaN       NaN   
                                              2022-11       NaN       NaN   
                                              2022-12  0.156125  0.261469   
                                              2023-01  0.162206  0.234956   
                                              2023-02  0.137381  0.210600   
                                              2023-03  0.121675  0.215300   
                                              2023-04  0.115706  0.215006   
                                              2023-05  0.142000  0.200450   
                                              2023-06  0.139125  0.171713   

                                                            b07     b08  \
obs_id                                        month                       
zimbabwe:0002e51c-5124-4614-bb1b-1c1ae307de80 2022-10       NaN     NaN   
                                              2022-11       NaN     NaN   
                                              2022-12  0.296106  0.2702   
                                              2023-01  0.271425  0.2232   
                                              2023-02  0.240400  0.2066   
                                              2023-03  0.243019  0.1917   
                                              2023-04  0.253325  0.2316   
                                              2023-05  0.227831  0.2208   
                                              2023-06  0.191862  0.1855   

                                                            b8a       b11  \
obs_id                                        month                         
zimbabwe:0002e51c-5124-4614-bb1b-1c1ae307de80 2022-10       NaN       NaN   
                                              2022-11       NaN       NaN   
                                              2022-12  0.328512  0.345562   
                                              2023-01  0.293025  0.349425   
                                              2023-02  0.263469  0.286738   
                                              2023-03  0.266813  0.277756   
                                              2023-04  0.277387  0.254575   
                                              2023-05  0.262569  0.290944   
                                              2023-06  0.219169  0.315387   

                                                            b12      ndvi  \
obs_id                                        month                         
zimbabwe:0002e51c-5124-4614-bb1b-1c1ae307de80 2022-10       NaN       NaN   
                                              2022-11       NaN       NaN   
                                              2022-12  0.261256  0.596927   
                                              2023-01  0.312794  0.403332   
                                              2023-02  0.212550  0.417010   
                                              2023-03  0.218881  0.522033   
                            

harvest_year,feature,2022,2023
0,b02,946,5459
1,b03,946,5459
2,b04,946,5459
3,b05,946,5459
4,b06,946,5459
5,b07,946,5459
6,b08,946,5459
7,b8a,946,5459
8,b11,946,5459
9,b12,946,5459


,year,family,field_months,present_vectors,dimensions,nonfinite_inside_vectors
0,2022,mosaiks_chip,1521,575,[1024],0
1,2022,mosaiks_field,1521,573,[1024],0
2,2023,mosaiks_chip,18873,13414,[1024],0
3,2023,mosaiks_field,18873,13288,[1024],0


coverage_state  clear  cloudy  no_scene
year month                             
2022 2021-10      169       0         0
     2021-11      169       0         0
     2021-12      169       0         0
     2022-01        0       0       169
     2022-02        0       0       169
     2022-03       68      83        18
     2022-04        0       0       169
     2022-05        0       0       169
     2022-06        0       0       169
2023 2022-10        0       0      2097
     2022-11        0       0      2097
     2022-12     2001      96         0
     2023-01     1673     424         0
     2023-02     1574     523         0
     2023-03     1875     222         0
     2023-04     2097       0         0
     2023-05     2097       0         0
     2023-06     2097       0         0

## 6. The project's modelling subset

Keep all records in the overview, but follow the existing loader's rule for modelling: `include_primary` and at least four clear months. Here this selects **1,880 records, all from the 2023 harvest**. All 169 records from 2022 are flagged `dropped_year_2022_imagery_gap`; the 2023 exclusions are 212 intercropped records and 5 irrigated records.

`X` contains all seasonal feature families on these rows; `X_indices` is the project's 10-feature baseline. `y` is kg/ha and `y_t_ha` is t/ha. Feature availability alone does not override the project's inclusion flags. For model fitting, use the project's frozen splits, fit preprocessing on training data, and check harvest timing if forecasting before harvest.


In [8]:
model_mask = zimbabwe.include_primary & zimbabwe.months_with_data.ge(4)
X = X_all.loc[model_mask].copy()
X_indices = X_families["season_indices"].loc[model_mask].copy()
y = y_all.loc[model_mask].copy()
y_t_ha = y_all_t_ha.loc[model_mask].copy()
assert X.index.equals(y.index) and X_indices.index.equals(y.index)
assert np.isfinite(X.to_numpy()).all() and np.isfinite(y).all()

# Verify that the baseline rows and target agree with the existing project loader.
import sys
sys.path.insert(0, str(ROOT / "src"))
from cropyield.data.xy import load_xy
baseline_X, baseline_y = load_xy(COHORT, ROOT / "data")
pd.testing.assert_frame_equal(X_indices, baseline_X)
pd.testing.assert_series_equal(y_t_ha, baseline_y)

display(pd.DataFrame([
    {"object": "X_all", "rows": len(X_all), "columns": X_all.shape[1], "scope": "all 2022 and 2023 records"},
    {"object": "X", "rows": len(X), "columns": X.shape[1], "scope": "project modelling subset"},
    {"object": "X_indices", "rows": len(X_indices), "columns": X_indices.shape[1], "scope": "baseline subset"},
    {"object": "y / y_t_ha", "rows": len(y), "columns": 1, "scope": "kg/ha / t/ha; aligned with X"}
]))
display(X_indices.head().join(y).join(y_t_ha))
print("Verified: unique observation keys, aligned X/y, finite model values, target units, and agreement with load_xy.")
con.close()


,object,rows,columns,scope
0,X_all,2266,2186,all 2022 and 2023 records
1,X,1880,2186,project modelling subset
2,X_indices,1880,10,baseline subset
3,y / y_t_ha,1880,1,kg/ha / t/ha; aligned with X


,ndvi_season_max,ndvi_season_mean,gcvi_season_max,gcvi_season_mean,mtci_season_max,mtci_season_mean,ndvi705_season_max,ndvi705_season_mean,ndvi740_season_max,ndvi740_season_mean,yield_kg_ha,yield_t_ha
obs_id,,,,,,,,,,,,
zimbabwe:0002e51c-5124-4614-bb1b-1c1ae307de80,0.596927,0.458077,3.195652,2.342303,2.029441,1.395117,0.333693,0.220610,0.048309,0.006750,1525.625,1.525625
zimbabwe:002b9802-b084-47a9-9ba6-0e87a828b826,0.844321,0.465507,6.100918,2.690319,8.761958,3.375413,0.659535,0.344775,0.166642,0.093019,3575.000,3.575000
zimbabwe:0052277a-e7f7-4964-b02b-f985f6f56a4e,0.519795,0.296590,2.172999,1.323875,19.469453,6.058138,0.359690,0.214768,0.099881,0.080257,4305.000,4.305000
zimbabwe:005b6c04-11ac-4dcb-9611-eb126a4d8ac9,0.726695,0.516804,3.954407,2.712628,3.093231,2.218613,0.466363,0.303710,0.137473,0.097301,1208.125,1.208125
zimbabwe:007192ec-f673-477d-9577-28b7b0c875c6,0.599426,0.412475,2.290788,1.623370,3.372808,2.883401,0.386736,0.270461,0.115006,0.074809,2380.000,2.380000


Verified: unique observation keys, aligned X/y, finite model values, target units, and agreement with load_xy.


All code outputs above are saved in this notebook. Run all cells with the project's Python environment to refresh them. Full values remain accessible as `zimbabwe`, `yields`, `X_all`, `X_families`, `monthly`, `X_monthly`, `X_monthly_vectors`, `nonimage`, `X`, and `y`; displayed tables are previews where indicated.
